# Master Script — DETECÇÃO DE ANOMALIAS (scikit-learn)

**Guia de consulta.** Não foi feito para rodar de ponta a ponta: é um repositório de
sintaxe, pipelines e grades de hiperparâmetros para copiar quando um projeto novo começa.

### Fluxo do documento
| # | Etapa | O que tem aqui |
|---|-------|----------------|
| 0 | Setup | Imports canônicos, `RANDOM_STATE`, config global |
| 1 | Data Prep | Transações fictícias com **três tipos de anomalia plantados** |
| 2 | Split | Estratificado, e a diferença entre *outlier* e *novelty detection* |
| 3 | Pipelines | Pré-processamento e um pipeline por família matemática |
| 4 | Validação | Torneio por PR-AUC, contaminação do treino e comparação com modelo supervisionado |
| 5 | Otimização | `RandomizedSearchCV` com rótulo e **estabilidade do top-k** sem rótulo |
| 6 | Prova Final | Limiar por capacidade e custo, recall por tipo, explicação dos alertas, persistência |
| A | Apêndice | Cheatsheet de famílias, sintomas e armadilhas |

### As cinco diferenças em relação à classificação
1. **O `fit` ignora o `y`.** O rótulo, quando existe, serve só para **avaliar e ajustar**.
2. **O produto é um ranking.** O detector entrega um escore; o limiar é uma decisão separada, ligada a quantos casos o time consegue investigar.
3. **O sinal do escore confunde.** No scikit-learn, `decision_function` **positivo é normal** e negativo é anomalia. Sem inverter o sinal, o ROC-AUC sai abaixo de 0,5.
4. **Acurácia não serve.** Com 4% de anomalias, chamar tudo de normal acerta 96%. Use **PR-AUC** e **precisão no top-k**.
5. **`contamination` não muda o ranking.** Ele só define onde `predict` corta; ajustar esse parâmetro não melhora o PR-AUC.

---
## 0. Setup e imports

In [ ]:
# --- Núcleo ---------------------------------------------------------------
import numpy as np
import pandas as pd

# --- Pré-processamento ----------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,      # sensível aos próprios outliers
    RobustScaler,        # mediana e IQR: padrão em detecção de anomalias
    OneHotEncoder, OrdinalEncoder,
)
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem

# --- Detectores: famílias matemáticas -------------------------------------
from sklearn.base import BaseEstimator, OutlierMixin, clone
from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier  # isolamento
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors            # densidade, distância
from sklearn.svm import OneClassSVM                                           # fronteira (kernel)
from sklearn.linear_model import SGDOneClassSVM                               # fronteira (escalável)
from sklearn.covariance import EllipticEnvelope                               # Mahalanobis robusta
from sklearn.mixture import GaussianMixture                                   # probabilístico
from sklearn.cluster import MiniBatchKMeans                                   # distância ao centróide
from sklearn.neural_network import MLPRegressor                               # autoencoder

# --- Split e busca --------------------------------------------------------
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_validate,
    RandomizedSearchCV, ParameterSampler,
)

# --- Métricas -------------------------------------------------------------
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, confusion_matrix,
    PrecisionRecallDisplay,
)
from sklearn.inspection import permutation_importance
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import ks_2samp

import matplotlib.pyplot as plt
import joblib

# --- Config global --------------------------------------------------------
import os
import time
import warnings

RANDOM_STATE = 42

# Paralelismo em DOIS níveis, de propósito. Cada worker do joblib abre um pool
# OpenMP do tamanho da máquina: com os dois em -1 surgem n² threads e o LightGBM
# e afins travam. O -1 fica só na validação cruzada e na busca.
N_JOBS       = -1
N_JOBS_MODEL = 1

# MASTER_SCRIPT_FAST=1 reduz tudo para os testes automatizados.
MODO_RAPIDO    = os.getenv("MASTER_SCRIPT_FAST") == "1"
N_LINHAS       = 500 if MODO_RAPIDO else 4_000   # linhas do DataFrame fictício
N_ITER_BUSCA   = 5   if MODO_RAPIDO else 40      # n_iter das buscas da etapa 5
N_RODADAS_ESTAB = 2  if MODO_RAPIDO else 5       # rodadas da estabilidade do top-k
N_REPEATS_PERM = 3   if MODO_RAPIDO else 10      # repetições da importância por permutação
MAX_ITER_MLP   = 100 if MODO_RAPIDO else 400     # épocas do autoencoder

# Parâmetros do problema
PREVALENCIA_ESPERADA = 0.04   # fração de anomalias que se acredita existir
CAPACIDADE           = 0.05   # fração dos casos que o time consegue investigar

---
## 1. Carregamento e Limpeza (Data Prep)

Transações fictícias de dois perfis (pessoas e empresas), com **anomalias plantadas de três
tipos**. O gabarito existe para conferir o que cada detector encontra, situação rara na prática.

| Tipo | Como foi plantada | Quem tende a achar |
|---|---|---|
| `global` | Valor e distância bem acima do normal | Quase todos, até o Z-score |
| `correlacao` | Valor normal no geral, mas 2 a 3,5 vezes a média da própria conta | Quem olha as variáveis em conjunto (Mahalanobis, PCA) |
| `contextual` | Valor alto de madrugada, com cada variável plausível isoladamente | Densidade e isolamento |

In [ ]:
# ==== 1.1 Leitura ==========================================================
# df = pd.read_csv("dados.csv", sep=",", decimal=".", encoding="utf-8")
# df = pd.read_parquet("dados.parquet")

# ---- DataFrame fictício (substituir pela leitura real) --------------------
n = N_LINHAS
rng = np.random.default_rng(RANDOM_STATE)

# Dois perfis normais: densidades diferentes, o caso em que o LOF faz diferença
empresa = rng.random(n) < 0.30
media_30d = np.where(empresa, rng.lognormal(np.log(900), 0.5, n), rng.lognormal(np.log(55), 0.6, n))

df = pd.DataFrame({
    "id_transacao": np.arange(n),
    "valor_medio_30d": media_30d,
    "valor": media_30d * rng.lognormal(0, 0.25, n),   # valor acompanha a média da conta
    "hora": np.where(empresa, rng.normal(11, 2.5, n), rng.normal(14, 4, n)) % 24,
    "transacoes_24h": np.where(empresa, rng.poisson(12, n), rng.poisson(2, n)).astype(float),
    "distancia_km": np.where(empresa, rng.exponential(3, n), rng.exponential(8, n)),
    "idade_conta_dias": rng.gamma(3, 300, n),
    "canal": rng.choice(["app", "web", "pos"], n, p=[.5, .2, .3]),
    "categoria": rng.choice(["varejo", "servicos", "viagem", "online", "outros"], n),
})

# Eventos raros, mas legítimos: é deles que vêm os alarmes falsos em dados reais.
# Sem isso, o problema fica fácil demais e todo detector parece perfeito.
raros = rng.choice(n, int(n * .06), replace=False)
df.loc[raros, "valor"] *= rng.uniform(1.5, 4, len(raros))
df.loc[raros, "distancia_km"] *= rng.uniform(3, 15, len(raros))
df.loc[raros, "hora"] = rng.uniform(0, 24, len(raros))

# Anomalias plantadas, em três tipos de igual tamanho
n_anomalias = int(n * PREVALENCIA_ESPERADA)
idx_anom = rng.choice(n, n_anomalias, replace=False)
grupos = np.array_split(idx_anom, 3)

df["tipo_anomalia"] = "normal"
df.loc[grupos[0], "tipo_anomalia"] = "global"
df.loc[grupos[1], "tipo_anomalia"] = "correlacao"
df.loc[grupos[2], "tipo_anomalia"] = "contextual"

# global: valor e distância muito acima de qualquer perfil
df.loc[grupos[0], "valor"] *= rng.uniform(6, 15, len(grupos[0]))
df.loc[grupos[0], "distancia_km"] = rng.uniform(80, 400, len(grupos[0]))

# correlacao: o valor sai do padrão da PRÓPRIA conta
df.loc[grupos[1], "valor"] = df.loc[grupos[1], "valor_medio_30d"] * rng.uniform(2, 3.5, len(grupos[1]))

# contextual: valor elevado, de madrugada e fora de casa
df.loc[grupos[2], "valor"] = df.loc[grupos[2], "valor_medio_30d"] * rng.uniform(1.5, 2.5, len(grupos[2]))
df.loc[grupos[2], "hora"] = rng.uniform(2, 5, len(grupos[2]))
df.loc[grupos[2], "distancia_km"] = rng.uniform(15, 60, len(grupos[2]))

df["anomalia"] = (df["tipo_anomalia"] != "normal").astype(int)

# Faltantes, para o imputador do pipeline ter o que fazer
df.loc[rng.choice(n, int(n * .02), replace=False), "distancia_km"] = np.nan
df.loc[rng.choice(n, int(n * .02), replace=False), "canal"] = np.nan

print(df.shape)
df.head()

In [ ]:
# ==== 1.2 Diagnóstico ======================================================
print(f"Prevalência de anomalias: {df['anomalia'].mean():.2%}")
display(df["tipo_anomalia"].value_counts().to_frame("linhas"))
display(df.describe().T.round(2))
print(df.isna().mean().mul(100).round(2).loc[lambda s: s > 0])

# Variáveis monetárias e de distância têm cauda longa: em escala bruta, poucas linhas
# dominam a distância. Por isso o log em 1.3.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for tipo, grupo in df.groupby("tipo_anomalia"):
    axes[0].scatter(np.log1p(grupo["valor_medio_30d"]), np.log1p(grupo["valor"]),
                    s=8 if tipo == "normal" else 28, alpha=.35 if tipo == "normal" else .9, label=tipo)
axes[0].set_xlabel("log(valor médio 30d)"); axes[0].set_ylabel("log(valor)")
axes[0].set_title("Anomalias de correlação escapam da diagonal"); axes[0].legend()
axes[1].hist(np.log1p(df["valor"]), bins=50, color="#4C72B0")
axes[1].set_xlabel("log(valor)"); axes[1].set_title("Olhando só o valor, boa parte delas some")
plt.tight_layout(); plt.show()

In [ ]:
# ==== 1.3 Limpeza padrão ==================================================
# a) Identificadores fora: id numérico vira "variável" e isola linhas por acaso
df = df.drop(columns=["id_transacao"], errors="ignore")

# b) Atenção: NÃO remova outliers aqui. Em detecção de anomalias, eles são o objetivo.

# c) Cauda longa em log. Transformação sem estado, então é seguro fazer antes do split.
LOG_COLS = ["valor", "valor_medio_30d", "distancia_km", "idade_conta_dias"]
for col in LOG_COLS:
    df[f"log_{col}"] = np.log1p(df[col])

# d) Hora é cíclica: 23h e 1h são vizinhas, e o número bruto diz o contrário
df["sin_hora"] = np.sin(2 * np.pi * df["hora"] / 24)
df["cos_hora"] = np.cos(2 * np.pi * df["hora"] / 24)

NUM_FEATURES = [f"log_{c}" for c in LOG_COLS] + ["sin_hora", "cos_hora", "transacoes_24h"]
CAT_FEATURES = ["canal", "categoria"]
df[CAT_FEATURES] = df[CAT_FEATURES].astype("category")

X = df[NUM_FEATURES + CAT_FEATURES]
y = df["anomalia"]            # usado só para AVALIAR: nenhum detector aprende com ele
tipo = df["tipo_anomalia"]    # usado só no diagnóstico da etapa 6
print(X.shape)

---
## 2. Separação dos Dados (Split)

Duas perguntas parecidas, com ferramentas diferentes:

| Modo | Pergunta | O treino contém anomalias? | Como no scikit-learn |
|---|---|---|---|
| **Outlier detection** | "O que é estranho *nesta* base?" | Sim | `LocalOutlierFactor().fit_predict(X)` |
| **Novelty detection** | "Esta linha *nova* é estranha?" | Idealmente não | `LocalOutlierFactor(novelty=True).fit(X)` e depois `.predict(novo)` |

> [!IMPORTANT]
> O split é **estratificado** para que o teste receba anomalias. Com dados reais de fraude, troque
> por um corte temporal, como no notebook 04: o golpe de amanhã nem sempre se parece com o de hoje.

In [ ]:
# ==== 2.1 Split estratificado =============================================
X_train, X_test, y_train, y_test, tipo_train, tipo_test = train_test_split(
    X, y, tipo, test_size=0.20, stratify=y, random_state=RANDOM_STATE,
)
print(f"treino: {X_train.shape} ({y_train.sum()} anomalias) | "
      f"teste: {X_test.shape} ({y_test.sum()} anomalias)")

In [ ]:
# ==== 2.2 Validação cruzada ===============================================
# O detector ignora o y no fit, mas os folds precisam de anomalias para medir o
# resultado. Por isso a estratificação continua obrigatória.
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

---
## 3. Construção dos Pipelines

| Receita | Numéricas | Categóricas | Modelos indicados |
|---|---|---|---|
| `"robust"` | mediana + `RobustScaler` | moda + `OneHotEncoder` | LOF, KNN, K-Means, SVM, PCA, autoencoder |
| `"num"` | mediana + `RobustScaler` | descartadas | Z-score, `EllipticEnvelope`, GMM |
| `"tree"` | mediana, sem escala | moda + `OrdinalEncoder` | `IsolationForest` |

O `RobustScaler` é o padrão porque as próprias anomalias inflam a média e o desvio do
`StandardScaler`. A receita `"num"` existe porque colunas one-hot deixam a matriz de covariância
singular, o que quebra os métodos baseados nela.

In [ ]:
# ==== 3.1 Fábrica de pré-processadores ====================================
def build_preprocessor(kind="robust", num=NUM_FEATURES, cat=CAT_FEATURES):
    num_escalada = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    if kind == "robust":
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
        return ColumnTransformer([("num", num_escalada, num), ("cat", cat_pipe, cat)])
    if kind == "num":
        return ColumnTransformer([("num", num_escalada, num)])
    if kind == "tree":
        num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
        return ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat)])
    raise ValueError(f"kind desconhecido: {kind!r}")

In [ ]:
# ==== 3.2 Detectores próprios, na mesma convenção do scikit-learn =========
# Convenção: decision_function positivo = normal, negativo = anomalia.
# Basta implementar _ajustar e _anomalia (quanto maior, mais anômalo).
class DetectorPorEscore(OutlierMixin, BaseEstimator):
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self._ajustar(X)
        self.offset_ = np.quantile(self._anomalia_treino(X), 1 - self.contamination)
        return self

    def _anomalia_treino(self, X):
        return self._anomalia(X)

    def decision_function(self, X):
        return self.offset_ - self._anomalia(np.asarray(X, dtype=float))

    def score_samples(self, X):
        return -self._anomalia(np.asarray(X, dtype=float))

    def predict(self, X):
        return np.where(self.decision_function(X) < 0, -1, 1)


class Aleatorio(DetectorPorEscore):
    # Baseline: o PR-AUC dele é a própria prevalência. Quem não supera isso não detecta nada.
    def __init__(self, contamination=PREVALENCIA_ESPERADA, random_state=RANDOM_STATE):
        self.contamination = contamination
        self.random_state = random_state

    def _ajustar(self, X):
        pass

    def _anomalia(self, X):
        return np.random.default_rng(self.random_state).random(len(X))


class ZScoreMaximo(DetectorPorEscore):
    # Baseline estatístico: o maior desvio entre as variáveis, uma de cada vez.
    # Espera-se que ele não veja anomalias de correlação.
    def __init__(self, contamination=PREVALENCIA_ESPERADA):
        self.contamination = contamination

    def _ajustar(self, X):
        pass

    def _anomalia(self, X):
        return np.abs(X).max(axis=1)


class DistanciaKNN(DetectorPorEscore):
    # Distância média aos k vizinhos. Simples, e costuma ir muito bem.
    def __init__(self, n_neighbors=10, contamination=PREVALENCIA_ESPERADA):
        self.n_neighbors = n_neighbors
        self.contamination = contamination

    def _ajustar(self, X):
        self.nn_ = NearestNeighbors(n_neighbors=self.n_neighbors, n_jobs=N_JOBS_MODEL).fit(X)

    def _anomalia_treino(self, X):
        return self.nn_.kneighbors()[0].mean(axis=1)   # sem X, cada ponto ignora a si mesmo

    def _anomalia(self, X):
        return self.nn_.kneighbors(X)[0].mean(axis=1)


class DistanciaCentroide(DetectorPorEscore):
    # Distância ao centróide mais próximo: barato, serve para milhões de linhas.
    def __init__(self, n_clusters=5, contamination=PREVALENCIA_ESPERADA, random_state=RANDOM_STATE):
        self.n_clusters = n_clusters
        self.contamination = contamination
        self.random_state = random_state

    def _ajustar(self, X):
        self.km_ = MiniBatchKMeans(n_clusters=self.n_clusters, n_init=5,
                                   random_state=self.random_state).fit(X)

    def _anomalia(self, X):
        return self.km_.transform(X).min(axis=1)


class DensidadeGMM(DetectorPorEscore):
    # Log-verossimilhança negativa: pouco provável sob a mistura de gaussianas = anômalo.
    def __init__(self, n_components=3, covariance_type="full", contamination=PREVALENCIA_ESPERADA,
                 random_state=RANDOM_STATE):
        self.n_components = n_components
        self.covariance_type = covariance_type
        self.contamination = contamination
        self.random_state = random_state

    def _ajustar(self, X):
        self.gmm_ = GaussianMixture(n_components=self.n_components, reg_covar=1e-3,
                                    covariance_type=self.covariance_type,
                                    random_state=self.random_state).fit(X)

    def _anomalia(self, X):
        return -self.gmm_.score_samples(X)


class ErroReconstrucaoPCA(DetectorPorEscore):
    # Erro ao comprimir e reconstruir: acha o que quebra as correlações da base.
    def __init__(self, n_components=4, contamination=PREVALENCIA_ESPERADA):
        self.n_components = n_components
        self.contamination = contamination

    def _ajustar(self, X):
        self.pca_ = PCA(n_components=self.n_components).fit(X)

    def _anomalia(self, X):
        recon = self.pca_.inverse_transform(self.pca_.transform(X))
        return ((X - recon) ** 2).sum(axis=1)


class Autoencoder(DetectorPorEscore):
    # A versão não-linear do PCA: uma rede treinada para reproduzir a própria entrada.
    def __init__(self, hidden_layer_sizes=(8, 4, 8), alpha=1e-3, learning_rate_init=1e-3,
                 max_iter=MAX_ITER_MLP, contamination=PREVALENCIA_ESPERADA, random_state=RANDOM_STATE):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.alpha = alpha
        self.learning_rate_init = learning_rate_init
        self.max_iter = max_iter
        self.contamination = contamination
        self.random_state = random_state

    def _ajustar(self, X):
        self.rede_ = MLPRegressor(hidden_layer_sizes=self.hidden_layer_sizes, alpha=self.alpha,
                                  learning_rate_init=self.learning_rate_init,
                                  max_iter=self.max_iter, random_state=self.random_state).fit(X, X)

    def _anomalia(self, X):
        return ((X - self.rede_.predict(X)) ** 2).sum(axis=1)


class EnsembleDeRanks(DetectorPorEscore):
    # Escores de detectores diferentes têm escalas incomparáveis. Converter cada um em
    # posição no treino (percentil) e tirar a média resolve, e costuma estabilizar o resultado.
    def __init__(self, detectores=None, contamination=PREVALENCIA_ESPERADA):
        self.detectores = detectores
        self.contamination = contamination

    def _ajustar(self, X):
        self.ajustados_ = [clone(d).fit(X) for d in self.detectores]
        self.referencias_ = [np.sort(-d.decision_function(X)) for d in self.ajustados_]

    def _anomalia(self, X):
        ranks = [np.searchsorted(ref, -d.decision_function(X)) / len(ref)
                 for d, ref in zip(self.ajustados_, self.referencias_)]
        return np.mean(ranks, axis=0)

In [ ]:
# ==== 3.3 Um Pipeline por família matemática ==============================
C = PREVALENCIA_ESPERADA

PIPELINES = {
    # ---------------- Baselines --------------------------------------------
    "Aleatorio": Pipeline([("prep", build_preprocessor("num")), ("model", Aleatorio())]),
    "ZScore":    Pipeline([("prep", build_preprocessor("num")), ("model", ZScoreMaximo())]),

    # ---------------- Isolamento: anomalias se separam com poucos cortes ----
    "IsolationForest": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", IsolationForest(n_estimators=200, contamination=C,
                                  random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL)),
    ]),

    # ---------------- Densidade local e distância ---------------------------
    "LOF": Pipeline([
        ("prep", build_preprocessor("robust")),
        ("model", LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=C, n_jobs=N_JOBS_MODEL)),
    ]),
    "KNN": Pipeline([("prep", build_preprocessor("robust")), ("model", DistanciaKNN())]),
    "KMeans": Pipeline([("prep", build_preprocessor("robust")), ("model", DistanciaCentroide())]),

    # ---------------- Fronteira: uma superfície envolve os dados normais ----
    "OneClassSVM": Pipeline([
        ("prep", build_preprocessor("robust")),
        ("model", OneClassSVM(kernel="rbf", gamma="scale", nu=C)),
    ]),
    "SGDOneClassSVM": Pipeline([
        ("prep", build_preprocessor("robust")),
        ("nystroem", Nystroem(n_components=100, random_state=RANDOM_STATE)),
        ("model", SGDOneClassSVM(nu=C, random_state=RANDOM_STATE)),
    ]),

    # ---------------- Estatístico: forma da distribuição --------------------
    "EllipticEnvelope": Pipeline([
        ("prep", build_preprocessor("num")),
        ("model", EllipticEnvelope(contamination=C, support_fraction=.8, random_state=RANDOM_STATE)),
    ]),
    "GMM": Pipeline([("prep", build_preprocessor("num")), ("model", DensidadeGMM())]),

    # ---------------- Reconstrução: o normal é fácil de comprimir ------------
    "PCA": Pipeline([("prep", build_preprocessor("robust")), ("model", ErroReconstrucaoPCA())]),
    "Autoencoder": Pipeline([("prep", build_preprocessor("robust")), ("model", Autoencoder())]),

    # ---------------- Ensemble ----------------------------------------------
    "EnsembleRanks": Pipeline([
        ("prep", build_preprocessor("robust")),
        ("model", EnsembleDeRanks(detectores=[
            IsolationForest(n_estimators=200, random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL),
            LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=N_JOBS_MODEL),
            ErroReconstrucaoPCA(),
        ])),
    ]),
}
print(f"Total de famílias registradas: {len(PIPELINES)}")

In [ ]:
# ==== 3.4 Outlier detection: rotular a própria base ========================
# Sem novelty=True, o LOF só tem fit_predict: serve para limpar ou auditar esta base,
# não para pontuar linhas novas. Chamar predict nesse modo levanta erro.
prep_auditoria = build_preprocessor("robust")
lof_auditoria = LocalOutlierFactor(n_neighbors=20, contamination=PREVALENCIA_ESPERADA, n_jobs=N_JOBS_MODEL)
rotulos = lof_auditoria.fit_predict(prep_auditoria.fit_transform(X_train))   # -1 = anomalia
print(f"linhas marcadas na própria base: {(rotulos == -1).sum()} de {len(rotulos)}")
print(f"LOF mais extremo: {-lof_auditoria.negative_outlier_factor_.min():.2f}  (perto de 1 = normal)")

---
## 4. Avaliação das Famílias

### Qual métrica escolher
| Métrica | O que responde | Quando usar |
|---|---|---|
| **PR-AUC** (`average_precision`) | Qualidade do ranking, focada na classe rara | Padrão. O acaso vale a prevalência |
| **ROC-AUC** | Chance de uma anomalia ficar acima de um normal | Comparar bases com prevalências diferentes; é otimista com classes raras |
| **Precisão no top-k** | Dos casos investigados, quantos eram reais? | Quando a capacidade do time é o limite |
| **Recall no top-k** | Das anomalias, quantas foram investigadas? | Quando deixar passar custa caro |

Todas usam o escore contínuo, então nenhuma depende de `contamination`.

In [ ]:
# ==== 4.1 Métricas avaliadas em cada fold =================================
# Scorers próprios por causa do sinal: anomalia = 1 no y, mas decision_function
# é MENOR para anomalias. Por isso o sinal negativo.
def escore_anomalia(modelo, X):
    return -modelo.decision_function(X)

def _indices_top_k(escore, fracao=CAPACIDADE):
    k = max(1, int(np.ceil(fracao * len(escore))))
    return np.argsort(escore)[-k:]

def pr_auc(modelo, X, y):
    return average_precision_score(y, escore_anomalia(modelo, X))

def roc_auc(modelo, X, y):
    return roc_auc_score(y, escore_anomalia(modelo, X))

def precisao_top_k(modelo, X, y):
    return np.asarray(y)[_indices_top_k(escore_anomalia(modelo, X))].mean()

def recall_top_k(modelo, X, y):
    y = np.asarray(y)
    return y[_indices_top_k(escore_anomalia(modelo, X))].sum() / max(y.sum(), 1)

SCORING = {
    "pr_auc": pr_auc,
    "roc_auc": roc_auc,
    "precisao_top_k": precisao_top_k,
    "recall_top_k": recall_top_k,
}
METRICA_ALVO = "pr_auc"

In [ ]:
# ==== 4.2 Torneio entre as famílias =======================================
def rodar_torneio(pipelines, X, y, cv=CV):
    linhas, falhas = [], {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        warnings.simplefilter("once", ConvergenceWarning)
        for nome, pipe in pipelines.items():
            t0 = time.perf_counter()
            try:
                res = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=N_JOBS,
                                     error_score="raise")
            except Exception as erro:
                falhas[nome] = f"{type(erro).__name__}: {erro}"
                print(f"{nome:<18} FALHOU: {falhas[nome]}")
                continue
            linha = {"modelo": nome}
            for m in SCORING:
                linha[f"{m}_mean"] = res[f"test_{m}"].mean()
                linha[f"{m}_std"] = res[f"test_{m}"].std()
            linha["fit_time_s"] = res["fit_time"].mean()
            linhas.append(linha)
            print(f"{nome:<18} PR-AUC={linha['pr_auc_mean']:.3f} (±{linha['pr_auc_std']:.3f})  "
                  f"ROC-AUC={linha['roc_auc_mean']:.3f}  {time.perf_counter() - t0:.1f}s")

    # Em dados reais, uma família falhar é aceitável. Nos testes, é defeito do notebook.
    if falhas and MODO_RAPIDO:
        raise RuntimeError(f"famílias falharam no torneio: {falhas}")
    return pd.DataFrame(linhas)

ranking = rodar_torneio(PIPELINES, X_train, y_train)
ranking = ranking.sort_values(f"{METRICA_ALVO}_mean", ascending=False).reset_index(drop=True)
display(ranking.round(3))

In [ ]:
# ==== 4.3 Leitura visual do torneio =======================================
top = ranking.iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(top["modelo"], top["pr_auc_mean"], xerr=top["pr_auc_std"], color="#4C72B0", alpha=.85)
axes[0].axvline(y_train.mean(), color="r", ls="--", label=f"Acaso = prevalência ({y_train.mean():.1%})")
axes[0].set_xlabel("PR-AUC"); axes[0].set_title("Qualidade do ranking (maior é melhor)")
axes[0].legend(); axes[0].grid(axis="x", alpha=.3)

axes[1].scatter(ranking["fit_time_s"], ranking["pr_auc_mean"], color="#55A868")
for _, linha in ranking.iterrows():
    axes[1].annotate(linha["modelo"], (linha["fit_time_s"], linha["pr_auc_mean"]), fontsize=8)
axes[1].set_xscale("log"); axes[1].set_xlabel("Tempo de ajuste (s)"); axes[1].set_ylabel("PR-AUC")
axes[1].set_title("Custo x qualidade"); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ==== 4.4 Escolha da família vencedora ====================================
BASELINES = {"Aleatorio", "ZScore"}
MELHOR_FAMILIA = ranking.loc[0, "modelo"]

melhor = ranking.loc[0]
empatados = ranking[ranking["pr_auc_mean"] >= melhor["pr_auc_mean"] - melhor["pr_auc_std"]]
print(f"Família vencedora: {MELHOR_FAMILIA}")
print(f"Empatadas dentro de 1 desvio-padrão: {list(empatados['modelo'])}")
print("No empate, prefira a mais barata e a mais fácil de explicar.")

if MELHOR_FAMILIA in BASELINES:
    print("\nATENÇÃO: um baseline venceu. O problema está nas variáveis, não no detector.")

In [ ]:
# ==== 4.5 Duas perguntas que mudam o projeto ==============================
# O teste continua guardado: as comparações usam uma validação retirada do treino.
X_sub, X_val, y_sub, y_val = train_test_split(
    X_train, y_train, test_size=.3, stratify=y_train, random_state=RANDOM_STATE,
)

# a) Quanto custa treinar com anomalias misturadas? (outlier x novelty)
linhas = []
for nome in ["IsolationForest", "LOF", "OneClassSVM", "PCA"]:
    contaminado = clone(PIPELINES[nome]).fit(X_sub)
    limpo = clone(PIPELINES[nome]).fit(X_sub[y_sub == 0])
    linhas.append({"modelo": nome,
                   "treino_com_anomalias": pr_auc(contaminado, X_val, y_val),
                   "treino_so_com_normais": pr_auc(limpo, X_val, y_val)})
display(pd.DataFrame(linhas).round(3))

# b) Se existe rótulo suficiente, um classificador costuma ganhar nos tipos já vistos.
#    O detector continua útil para o que nunca apareceu no histórico.
prep_sup = build_preprocessor("tree")
supervisionado = Pipeline([
    ("prep", prep_sup),
    ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
]).fit(X_sub, y_sub)
ap_sup = average_precision_score(y_val, supervisionado.predict_proba(X_val)[:, 1])
ap_det = pr_auc(clone(PIPELINES[MELHOR_FAMILIA]).fit(X_sub), X_val, y_val)
print(f"PR-AUC supervisionado (usa rótulo): {ap_sup:.3f}")
print(f"PR-AUC {MELHOR_FAMILIA} (ignora rótulo): {ap_det:.3f}")

---
## 5. Escolha e Otimização (`RandomizedSearchCV`)

Depende de haver rótulo:

| Situação | Como ajustar |
|---|---|
| **Há rótulos, mesmo poucos** | `RandomizedSearchCV` com os scorers de 4.1 (seção 5.3) |
| **Não há rótulo nenhum** | Prefira a configuração cujo top-k **se repete** entre amostras (seção 5.4) |

Estabilidade não prova acerto: ela apenas descarta configurações que mudam a cada amostra.

In [ ]:
# ==== 5.1 Grades de hiperparâmetros: repositório completo =================
from scipy.stats import loguniform, uniform, randint

GRIDS = {
"ZScore": {
    "prep__num__scaler": [RobustScaler(), StandardScaler()],
},
"IsolationForest": {
    "model__n_estimators": randint(100, 500),
    "model__max_samples":  [64, 128, 256, "auto"],   # amostras PEQUENAS costumam isolar melhor
    "model__max_features": uniform(0.5, 0.5),
},
"LOF": {
    "model__n_neighbors": randint(5, 60),
    "model__metric":      ["minkowski", "manhattan"],
},
"KNN": {
    "model__n_neighbors": randint(2, 40),
},
"KMeans": {
    "model__n_clusters": randint(2, 15),
},
"OneClassSVM": {
    "model__nu":    uniform(0.01, 0.2),
    "model__gamma": loguniform(1e-3, 1e0),
},
"SGDOneClassSVM": {
    "nystroem__gamma":        loguniform(1e-3, 1e0),
    "nystroem__n_components": randint(50, 150),
    "model__nu":              uniform(0.01, 0.2),
},
"EllipticEnvelope": {
    "model__support_fraction": uniform(0.6, 0.35),
},
"GMM": {
    "model__n_components":    randint(1, 6),
    "model__covariance_type": ["full", "diag", "tied"],
},
"PCA": {
    "model__n_components": randint(2, 8),
},
"Autoencoder": {
    "model__hidden_layer_sizes": [(4,), (8,), (8, 4, 8), (16, 4, 16)],
    "model__alpha":              loguniform(1e-5, 1e-1),
    "model__learning_rate_init": loguniform(1e-4, 1e-2),
},
}

In [ ]:
# ==== 5.2 Grade do bloco de pré-processamento ==============================
# Só vale para pipelines cujas numéricas passam por um passo "scaler".
GRID_PREP = {
    "prep__num__scaler": [RobustScaler(), StandardScaler()],
}

In [ ]:
# ==== 5.3 A busca, no pipeline vencedor, só com dados de TREINO ===========
pipe_vencedor = PIPELINES[MELHOR_FAMILIA]
grid_vencedor = dict(GRIDS.get(MELHOR_FAMILIA, {}))
if "prep__num__scaler" in pipe_vencedor.get_params():
    grid_vencedor.update(GRID_PREP)

if not grid_vencedor:
    print(f"{MELHOR_FAMILIA} não tem grade: a busca apenas reavalia a configuração atual.")

search = RandomizedSearchCV(
    estimator=pipe_vencedor,
    param_distributions=grid_vencedor,
    n_iter=N_ITER_BUSCA if grid_vencedor else 1,
    scoring=SCORING,
    refit=METRICA_ALVO,
    cv=CV,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
)
search.fit(X_train, y_train)

print(f"Melhor PR-AUC na validação cruzada: {search.best_score_:.3f}")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")

best_model = search.best_estimator_

colunas = ["rank_test_pr_auc", "mean_test_pr_auc", "std_test_pr_auc", "mean_test_precisao_top_k", "mean_fit_time"]
display(pd.DataFrame(search.cv_results_).sort_values("rank_test_pr_auc")[colunas].head().round(3))

In [ ]:
# ==== 5.4 Sem rótulo: estabilidade do top-k ================================
# Ajusta em duas subamostras, pontua a mesma base e mede quanto os dois top-k coincidem
# (índice de Jaccard: 1 = idênticos). Note que o y não aparece aqui.
def estabilidade_top_k(pipe, X, frac=.7, n_rodadas=N_RODADAS_ESTAB, seed=RANDOM_STATE):
    valores = []
    for i in range(n_rodadas):
        a = X.sample(frac=frac, random_state=seed + 2 * i)
        b = X.sample(frac=frac, random_state=seed + 2 * i + 1)
        top_a = set(_indices_top_k(escore_anomalia(clone(pipe).fit(a), X)))
        top_b = set(_indices_top_k(escore_anomalia(clone(pipe).fit(b), X)))
        valores.append(len(top_a & top_b) / len(top_a | top_b))
    return np.mean(valores)

grade_estab = GRIDS.get(MELHOR_FAMILIA) or GRIDS["IsolationForest"]
pipe_estab = PIPELINES[MELHOR_FAMILIA if MELHOR_FAMILIA in GRIDS else "IsolationForest"]

linhas = []
for params in ParameterSampler(grade_estab, n_iter=min(N_ITER_BUSCA, 5), random_state=RANDOM_STATE):
    candidato = clone(pipe_estab).set_params(**params)
    linhas.append({**{k.split("__")[-1]: v for k, v in params.items()},
                   "estabilidade_top_k": estabilidade_top_k(candidato, X_train)})
display(pd.DataFrame(linhas).sort_values("estabilidade_top_k", ascending=False).round(3))

---
## 6. A Prova Final (Inferência no Teste)

O teste foi preservado até aqui. O limiar é escolhido em **escores fora da amostra do treino**,
e o teste serve apenas para reportar o resultado.

In [ ]:
# ==== 6.1 Qualidade do ranking no teste ===================================
escore_teste = escore_anomalia(best_model, X_test)

resultado = pd.Series({metrica: funcao(best_model, X_test, y_test) for metrica, funcao in SCORING.items()})
display(resultado.round(3).to_frame("teste"))
print(f"Prevalência no teste (PR-AUC do acaso): {y_test.mean():.3f}")
print(f"CV (treino): {search.best_score_:.3f} | Teste: {resultado['pr_auc']:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
PrecisionRecallDisplay.from_predictions(y_test, escore_teste, ax=axes[0], name=MELHOR_FAMILIA)
axes[0].axhline(y_test.mean(), color="r", ls="--", label="Acaso"); axes[0].legend()
axes[0].set_title("Curva Precisão x Recall")
axes[1].hist(escore_teste[y_test == 0], bins=40, alpha=.7, density=True, label="Normal", color="#4C72B0")
axes[1].hist(escore_teste[y_test == 1], bins=15, alpha=.7, density=True, label="Anomalia", color="#C44E52")
axes[1].set_xlabel("Escore de anomalia"); axes[1].set_title("Separação dos escores"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ==== 6.2 O limiar: decisão de negócio, escolhida sem tocar no teste ======
# cross_val_predict não serve aqui: ele exige classes_, que detectores não têm.
escore_oof = np.empty(len(X_train))
for idx_ajuste, idx_fora in CV.split(X_train, y_train):
    modelo_fold = clone(best_model).fit(X_train.iloc[idx_ajuste])
    escore_oof[idx_fora] = escore_anomalia(modelo_fold, X_train.iloc[idx_fora])

# a) Por capacidade: o time investiga no máximo CAPACIDADE dos casos
limiar_capacidade = np.quantile(escore_oof, 1 - CAPACIDADE)

# b) Por custo: deixar passar uma anomalia custa muito mais que um alarme falso
CUSTO_FN, CUSTO_FP = 50, 1
candidatos = np.unique(np.quantile(escore_oof, np.linspace(.5, .999, 200)))
custos = [CUSTO_FN * ((escore_oof < t) & (y_train == 1)).sum()
          + CUSTO_FP * ((escore_oof >= t) & (y_train == 0)).sum() for t in candidatos]
limiar_custo = candidatos[int(np.argmin(custos))]

linhas = []
for nome, limiar in {"capacidade": limiar_capacidade, "custo": limiar_custo}.items():
    alerta = (escore_teste >= limiar).astype(int)
    linhas.append({"criterio": nome, "limiar": limiar, "alertas": alerta.sum(),
                   "precisao": precision_score(y_test, alerta, zero_division=0),
                   "recall": recall_score(y_test, alerta, zero_division=0)})
display(pd.DataFrame(linhas).round(3))

LIMIAR = limiar_capacidade
alerta_teste = (escore_teste >= LIMIAR).astype(int)
display(pd.DataFrame(confusion_matrix(y_test, alerta_teste),
                     index=["real: normal", "real: anomalia"],
                     columns=["sem alerta", "alerta"]))

In [ ]:
# ==== 6.3 O que o detector acha, por tipo de anomalia =====================
# O recall geral esconde pontos cegos. Só é possível abrir assim porque o gabarito existe;
# na prática, faça isso com os casos já confirmados pelo time.
por_tipo = (
    pd.DataFrame({"tipo": tipo_test.values, "alerta": alerta_teste})
    .groupby("tipo")["alerta"].agg(["size", "mean"])
    .rename(columns={"size": "linhas", "mean": "fracao_com_alerta"})
)
display(por_tipo.round(3))

In [ ]:
# ==== 6.4 Quais variáveis movem o ranking =================================
perm = permutation_importance(
    best_model, X_test, y_test, scoring=SCORING[METRICA_ALVO],
    n_repeats=N_REPEATS_PERM, random_state=RANDOM_STATE, n_jobs=N_JOBS,
)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
imp.plot(kind="barh", figsize=(9, 5), color="#4C72B0", alpha=.85)
plt.title("Importância por permutação (queda no PR-AUC)")
plt.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# ==== 6.5 Explicar cada alerta ============================================
# Quem investiga precisa saber POR QUE a linha subiu. Uma explicação simples e agnóstica ao
# modelo: quais variáveis mais fogem da mediana do treino, em unidades de IQR.
mediana = X_train[NUM_FEATURES].median()
iqr = (X_train[NUM_FEATURES].quantile(.75) - X_train[NUM_FEATURES].quantile(.25)).replace(0, 1)

def explicar(linha, n=2):
    desvios = ((linha[NUM_FEATURES].astype(float) - mediana) / iqr).dropna()
    maiores = desvios.abs().sort_values(ascending=False).index[:n]
    return ", ".join(f"{col} ({desvios[col]:+.1f} IQR)" for col in maiores)

ordem = np.argsort(escore_teste)[::-1][:8]
alertas = pd.DataFrame({
    "escore": escore_teste[ordem],
    "tipo_real": tipo_test.values[ordem],
    "motivo": [explicar(X_test.iloc[i]) for i in ordem],
})
display(alertas.round(3))
# Limitação: desvio marginal não explica anomalias de correlação, em que cada variável é normal.

In [ ]:
# ==== 6.6 Monitoramento: o detector continua válido? ======================
# Sem rótulo em produção, acompanhe a própria saída do modelo: taxa de alertas e
# distribuição dos escores. Mudança brusca indica drift, ou um ataque novo.
# A referência e os dados novos precisam ser pontuados pelo MESMO modelo: comparar escores
# de modelos diferentes acusa drift onde não há. Usamos o último modelo de 6.2 e o trecho
# do treino que ele não viu.
referencia = escore_anomalia(modelo_fold, X_train.iloc[idx_fora])
estatistica, p_valor = ks_2samp(referencia, escore_anomalia(modelo_fold, X_test))
print(f"Taxa de alertas: {alerta_teste.mean():.2%}  (planejada: {CAPACIDADE:.2%})")
print(f"Teste KS, escores do treino x novos: estatística={estatistica:.3f}, p={p_valor:.3f}")

In [ ]:
# ==== 6.7 Modelo final e persistência =====================================
# Refit com todos os dados. O limiar acompanha o modelo: sem ele, o escore não vira decisão.
best_model.fit(X, y)
ARQUIVO_MODELO = "modelo_anomalias.joblib"
joblib.dump({"modelo": best_model, "limiar": LIMIAR, "versao_sklearn": __import__("sklearn").__version__},
            ARQUIVO_MODELO)

artefato = joblib.load(ARQUIVO_MODELO)
novas = X.sample(5, random_state=RANDOM_STATE)
escores = -artefato["modelo"].decision_function(novas)
display(pd.DataFrame({"escore": escores, "alerta": escores >= artefato["limiar"]}, index=novas.index).round(3))

---
## Apêndice — Cheatsheet

### Qual família tentar primeiro
| Situação | Comece por | Por quê |
|---|---|---|
| Ponto de partida geral | **IsolationForest** | Rápido, pouco sensível a escala e com poucos parâmetros |
| Grupos com densidades diferentes | **LOF** ou **KNN** | Comparam cada ponto com a própria vizinhança |
| Anomalias que quebram correlações | **PCA** ou **EllipticEnvelope** | Medem o desvio em relação à estrutura conjunta |
| Milhões de linhas | **SGDOneClassSVM** ou **KMeans** | Escalam quase linearmente |
| Dados normais limpos disponíveis | **OneClassSVM** ou **Autoencoder** | Aprendem bem a fronteira do normal, mas sofrem com treino contaminado |
| Muitos rótulos confiáveis | **Classificador** do notebook 01 | Usar o rótulo costuma render mais nos tipos já vistos |
| Quer reduzir o risco da escolha | **EnsembleRanks** | Combina detectores que erram de formas diferentes |

### Sintomas e correções
| Sintoma | Causa provável | Correção |
|---|---|---|
| ROC-AUC bem abaixo de 0,5 | Sinal do escore invertido | Use `-decision_function` |
| PR-AUC perto da prevalência | Variáveis não carregam o sinal | Crie razões e comparações com o histórico da conta |
| Mudar `contamination` não altera o PR-AUC | Comportamento esperado | Ele só muda o corte de `predict`; ajuste o limiar na etapa 6 |
| `AttributeError` no `predict` do LOF | Faltou `novelty=True` | Use `novelty=True` para pontuar linhas novas |
| Uma variável domina os alertas | Escala ausente ou cauda longa | `RobustScaler` e log |
| Erro de covariância singular | One-hot em método baseado em covariância | Use a receita `"num"` |
| Alertas demais depois de um tempo | Drift | Monitore a taxa (6.6) e reajuste |

### Erros que invalidam o resultado
1. **Remover outliers na limpeza**, eliminando justamente o que se quer detectar.
2. **Usar acurácia** para avaliar uma classe rara.
3. **Escolher o limiar no teste**, o que torna a métrica final otimista.
4. **Treinar um detector de novidade com dados contaminados** sem medir o impacto (4.5).
5. **Ajustar o escalonador fora do Pipeline**, vazando informação da validação.
6. **Tratar alerta como confirmação.** Detector aponta o que é raro, e nem todo raro é problema.